# Gold Layer — Business Transformations

Reads from `project.gold.delivery_delay` (training data) and `project.gold.delivery_delay_predictions` (model output) to build aggregated Gold tables for dashboarding.

**Output tables:**
- `project.gold.agg_late_by_state` — geographic delivery risk
- `project.gold.agg_late_by_category` — product category risk
- `project.gold.agg_late_by_month` — seasonal patterns
- `project.gold.agg_high_risk_orders` — high-risk order profiles
- `project.gold.agg_model_performance` — predicted vs actual summary

---

## 1. Setup & Load Tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

CATALOG = "project"
GOLD    = "gold"

# Source tables
SOURCE_TABLE      = f"{CATALOG}.{GOLD}.delivery_delay"
PREDICTIONS_TABLE = f"{CATALOG}.{GOLD}.predictions"

df      = spark.read.table(SOURCE_TABLE)
pred_df = spark.read.table(PREDICTIONS_TABLE)

print(f"delivery_delay rows       : {df.count():,}")
print(f"predictions rows          : {pred_df.count():,}")
display(df.limit(3))

delivery_delay rows       : 106,225
predictions rows          : 21,429


order_id,customer_state,product_category_name_english,price,freight_value,installments,order_dow,order_hour,order_month,estimated_days,is_late
0010b2e5201cc5f1ae7e9c6cc8f5bd00,RJ,cool_stuff,48.9,16.6,3,2,17,9,16,0
0028de0ca693a1bb26448916a81105cc,RS,construction_tools_lights,29.99,15.31,1,4,14,8,27,0
0032d07457ae9c806c79368d7d9ce96b,RJ,housewares,159.0,27.19,2,7,18,3,38,1


## 2. Helper — Write Gold Delta Table

In [0]:
def write_gold(agg_df, table_name, zorder_col=None):
    full = f"{CATALOG}.{GOLD}.{table_name}"
    (
        agg_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full)
    )
    if zorder_col:
        spark.sql(f"OPTIMIZE {full} ZORDER BY ({zorder_col})")
    print(f"✓ Saved {full}  ({agg_df.count():,} rows)")

---
## 3. Aggregation 1 — Late Delivery Rate by State

**Insight from data:** AL (20.8%), MA (18.1%), SE (16.8%) are the worst states — more than 2.5x the national average.
SP has the most orders (44k) but only ~8% late. Geographic disparity is the #1 operational signal.

In [0]:
display(df.groupBy("customer_state").agg(F.round(F.avg("is_late") * 100, 2).alias("late_rate_pct")).select("late_rate_pct").agg(F.max("late_rate_pct").alias("max_late_rate_pct")))

max_late_rate_pct
20.77


## Risk Tier Classification for State-Level Late Delivery Analysis

Based on exploratory analysis of late delivery rates across all customer states, we observed:
- **Maximum late rate**: 20.77%
- **Distribution range**: 0% to 20.77%

**Risk Tier Thresholds:**
- **High Risk**: Late rate ≥ 15% — States with chronic delivery delays affecting 1 in 6-7 orders
- **Medium Risk**: Late rate ≥ 8% but < 15% — States with moderate delivery challenges
- **Low Risk**: Late rate < 8% — States with acceptable delivery performance

**Rationale:**
The 15% threshold captures the upper quartile of performance variance and identifies states requiring immediate operational intervention. The 8% threshold balances sensitivity to moderate issues while avoiding false positives in states with generally good performance. These thresholds are empirically grounded in the observed distribution and align with operational significance.

In [0]:
agg_state = (
    df.groupBy("customer_state")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("is_late").alias("late_orders"),
        F.round(F.avg("is_late") * 100, 2).alias("late_rate_pct"),
        F.round(F.avg("estimated_days"), 1).alias("avg_estimated_days"),
        F.round(F.avg("freight_value"), 2).alias("avg_freight"),
        F.round(F.sum("price"), 2).alias("total_revenue")
    )
    .withColumn("risk_tier",
        F.when(F.col("late_rate_pct") >= 15, "High")
         .when(F.col("late_rate_pct") >= 8,  "Medium")
         .otherwise("Low")
    )
    .orderBy(F.col("late_rate_pct").desc())
)

display(agg_state)


customer_state,total_orders,late_orders,late_rate_pct,avg_estimated_days,avg_freight,total_revenue,risk_tier
AL,414,86,20.77,33.1,35.93,76002.17,High
MA,778,141,18.12,31.5,38.6,114655.22,High
SE,363,61,16.8,31.4,36.84,55545.19,High
CE,1398,193,13.81,32.0,32.73,216848.68,Medium
PI,511,70,13.7,30.9,39.35,83623.79,Medium
BA,3559,428,12.03,30.2,26.4,479888.08,Medium
RJ,13599,1585,11.66,27.1,20.95,1699214.93,Medium
RR,44,5,11.36,46.9,42.28,6721.47,Medium
PA,1029,115,11.18,38.0,35.77,172413.4,Medium
PB,572,63,11.01,33.6,43.19,110134.4,Medium


In [0]:
write_gold(agg_state, "agg_late_by_state", zorder_col="customer_state")

✓ Saved project.gold.agg_late_by_state  (27 rows)


---
## 4. Aggregation 2 — Late Rate by Product Category

**Insight from data:** `audio` (11.9%), `home_comfort` (9.7%), `baby` (7.8%) are highest risk.
Categories like `baby` are especially critical — delay impacts customer trust significantly.

In [0]:
from pyspark.sql import functions as F
import statistics

# 1. VOLUME THRESHOLD (minimum orders per category)
vol_stats = df.groupBy("product_category_name_english").agg(F.count("*").alias("total_orders")).select("total_orders").collect()
volumes = [row.total_orders for row in vol_stats]
vol_median = statistics.median(volumes)
vol_q1 = statistics.quantiles(volumes, n=4)[0]
vol_iqr = statistics.quantiles(volumes, n=4)[2] - vol_q1
vol_threshold = vol_median - 1.5 * vol_iqr  # Lower bound for minimum volume

print(f"Volume - Median: {vol_median}, Q1: {vol_q1}, IQR: {vol_iqr}, Min Threshold: {vol_threshold}")

# 2. LATE RATE THRESHOLD (delivery risk)
late_stats = df.groupBy("product_category_name_english").agg(F.round(F.avg("is_late") * 100, 2).alias("late_rate_pct")).select("late_rate_pct").collect()
late_rates = [row.late_rate_pct for row in late_stats if row.late_rate_pct is not None]
late_median = statistics.median(late_rates)
late_q3 = statistics.quantiles(late_rates, n=4)[2]
late_iqr = late_q3 - statistics.quantiles(late_rates, n=4)[0]
late_threshold = late_median + 1.5 * late_iqr

print(f"Late Rate - Median: {late_median}, IQR: {late_iqr}, High Risk Threshold: {late_threshold}")

# 3. REVENUE THRESHOLD (business significance)
rev_stats = df.groupBy("product_category_name_english").agg(F.round(F.sum("price"), 2).alias("total_revenue")).select("total_revenue").collect()
revenues = [row.total_revenue for row in rev_stats if row.total_revenue is not None]
rev_median = statistics.median(revenues)
rev_q3 = statistics.quantiles(revenues, n=4)[2]
rev_iqr = rev_q3 - statistics.quantiles(revenues, n=4)[0]
rev_threshold = rev_median + 1.5 * rev_iqr

print(f"Revenue - Median: {rev_median}, IQR: {rev_iqr}, High Revenue Threshold: {rev_threshold}")

Volume - Median: 271.5, Q1: 83.5, IQR: 1710.75, Min Threshold: -2294.625
Late Rate - Median: 5.975, IQR: 2.92, High Risk Threshold: 10.355
Revenue - Median: 45719.07, IQR: 191034.43250000002, High Revenue Threshold: 332270.71875000006


The min threshold for volume is in negatives. this shows that the data for volume distribution is heavily right-skewed. so instead, we calculate the percentiles and choose a practical threshold 

In [0]:
# Look at distribution percentiles
vol_percentiles = df.groupBy("product_category_name_english").agg(F.count("*").alias("total_orders")).select("total_orders").collect()
volumes_sorted = sorted([row.total_orders for row in vol_percentiles])

print(f"Min: {volumes_sorted[0]}, 25th: {volumes_sorted[len(volumes_sorted)//4]}, Median: {volumes_sorted[len(volumes_sorted)//2]}, 75th: {volumes_sorted[3*len(volumes_sorted)//4]}, Max: {volumes_sorted[-1]}")

Min: 2, 25th: 94, Median: 275, 75th: 1849, Max: 10355


In [0]:
vol_threshold = 100

In [0]:
agg_cat = (
    df.groupBy("product_category_name_english")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("is_late").alias("late_orders"),
        F.round(F.avg("is_late") * 100, 2).alias("late_rate_pct"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.avg("freight_value"), 2).alias("avg_freight"),
        F.round(
            F.avg("freight_value") / (F.avg("price") + F.lit(0.01)) * 100, 2
        ).alias("freight_to_price_pct")
    )
    .filter(F.col("total_orders") >= vol_threshold)
    .withColumn("revenue_risk_flag",
        F.when(
            (F.col("late_rate_pct") >= late_threshold) & (F.col("total_revenue") >= rev_threshold),
            "High Revenue + High Risk"
        ).when(F.col("late_rate_pct") >= late_threshold, "High Risk")
         .when(F.col("total_revenue") >= rev_threshold, "High Revenue")
         .otherwise("Normal")
    )
    .orderBy(F.col("late_rate_pct").desc())
)

display(agg_cat)

product_category_name_english,total_orders,late_orders,late_rate_pct,total_revenue,avg_price,avg_freight,freight_to_price_pct,revenue_risk_flag
audio,354,42,11.86,50123.2,141.59,15.75,11.12,High Risk
fashion_underwear_beach,118,12,10.17,9036.85,76.58,14.58,19.04,Normal
christmas_supplies,148,15,10.14,8644.17,58.41,21.22,36.32,Normal
home_confort,413,40,9.69,55606.35,134.64,19.64,14.59,Normal
office_furniture,1630,128,7.85,261743.79,160.58,40.11,24.98,Normal
baby,2901,226,7.79,391835.28,135.07,22.32,16.52,High Revenue
electronics,2685,205,7.64,152781.74,56.9,16.78,29.49,Normal
health_beauty,9183,698,7.6,1202974.41,131.0,18.97,14.48,High Revenue
unknown_category,1502,114,7.59,170708.72,113.65,17.63,15.51,Normal
construction_tools_lights,294,22,7.48,38631.12,131.4,24.65,18.76,Normal


## Category Risk Classification – Statistical Method

Thresholds derived from distribution analysis (median ± 1.5×IQR):

- **Minimum Volume**: ≥ {vol_threshold:.0f} orders — Excludes low-volume categories with unreliable metrics
- **High Risk**: Late rate ≥ {late_threshold:.2f}% — Categories with statistically significant delivery delays
- **High Revenue**: Total revenue ≥ ${rev_threshold:,.0f} — Business-critical categories requiring priority

Categories flagged "High Revenue + High Risk" warrant immediate operational focus.

In [0]:
write_gold(agg_cat, "agg_late_by_category", zorder_col="product_category_name_english")

✓ Saved project.gold.agg_late_by_category  (52 rows)


---
## 5. Aggregation 3 — Seasonal Late Rate by Month

**Insight from data:** Feb (11.9%), Mar (14.6%), Nov (12%) are peak delay months.
Jun is the safest (1.8%). March is nearly 8x safer than June — clear seasonal capacity strain.

In [0]:
month_stats = df.groupBy("order_month").agg(F.round(F.avg("is_late") * 100, 2).alias("late_rate_pct")).select("late_rate_pct").collect()
late_rates = [row.late_rate_pct for row in month_stats]

import statistics
month_median = statistics.median(late_rates)
month_q3 = statistics.quantiles(late_rates, n=4)[2]
month_q1 = statistics.quantiles(late_rates, n=4)[0]
month_iqr = month_q3 - month_q1
month_threshold = month_median + 1.5 * month_iqr

print(f"Month threshold: {month_threshold}")

Month threshold: 15.145


In [0]:
month_name_map = F.create_map(
    F.lit(1),  F.lit("Jan"),  F.lit(2),  F.lit("Feb"),
    F.lit(3),  F.lit("Mar"),  F.lit(4),  F.lit("Apr"),
    F.lit(5),  F.lit("May"),  F.lit(6),  F.lit("Jun"),
    F.lit(7),  F.lit("Jul"),  F.lit(8),  F.lit("Aug"),
    F.lit(9),  F.lit("Sep"),  F.lit(10), F.lit("Oct"),
    F.lit(11), F.lit("Nov"),  F.lit(12), F.lit("Dec")
)

agg_month = (
    df.groupBy("order_month")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("is_late").alias("late_orders"),
        F.round(F.avg("is_late") * 100, 2).alias("late_rate_pct"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("estimated_days"), 1).alias("avg_estimated_days")
    )
    .withColumn("month_name", month_name_map[F.col("order_month")])
    .withColumn("is_peak_delay_month",
        F.when(F.col("late_rate_pct") >= month_threshold, F.lit(1)).otherwise(F.lit(0))
    )
    .orderBy("order_month")
)

display(agg_month)


order_month,total_orders,late_orders,late_rate_pct,total_revenue,avg_estimated_days,month_name,is_peak_delay_month
1,8613,469,5.45,999501.49,28.3,Jan,0
2,9004,1069,11.87,1027512.42,26.7,Feb,0
3,10563,1546,14.64,1278454.93,23.5,Mar,0
4,10064,466,4.63,1272989.09,25.2,Apr,0
5,11388,600,5.27,1421044.8,24.0,May,0
6,10136,183,1.81,1242859.16,27.1,Jun,0
7,10964,351,3.2,1309082.1,21.8,Jul,0
8,11535,550,4.77,1351820.54,19.2,Aug,0
9,4515,204,4.52,582917.61,23.2,Sep,0
10,5292,202,3.82,666854.21,25.4,Oct,0


In [0]:
write_gold(agg_month, "genuine_problematic_months")

✓ Saved project.gold.genuine_problematic_months  (12 rows)


In [0]:
month_threshold = statistics.quantiles(late_rates, n=4)[2]  # 75th percentile

worst_month = (
    df.groupBy("order_month")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("is_late").alias("late_orders"),
        F.round(F.avg("is_late") * 100, 2).alias("late_rate_pct"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("estimated_days"), 1).alias("avg_estimated_days")
    )
    .withColumn("month_name", month_name_map[F.col("order_month")])
    .withColumn("is_peak_delay_month",
        F.when(F.col("late_rate_pct") >= month_threshold, F.lit(1)).otherwise(F.lit(0))
    )
    .orderBy("order_month")
)

display(worst_month)


order_month,total_orders,late_orders,late_rate_pct,total_revenue,avg_estimated_days,month_name,is_peak_delay_month
1,8613,469,5.45,999501.49,28.3,Jan,0
2,9004,1069,11.87,1027512.42,26.7,Feb,1
3,10563,1546,14.64,1278454.93,23.5,Mar,1
4,10064,466,4.63,1272989.09,25.2,Apr,0
5,11388,600,5.27,1421044.8,24.0,May,0
6,10136,183,1.81,1242859.16,27.1,Jun,0
7,10964,351,3.2,1309082.1,21.8,Jul,0
8,11535,550,4.77,1351820.54,19.2,Aug,0
9,4515,204,4.52,582917.61,23.2,Sep,0
10,5292,202,3.82,666854.21,25.4,Oct,0


In [0]:
write_gold(agg_month, "worst_performing_months")

✓ Saved project.gold.worst_performing_months  (12 rows)


---
## 6. Aggregation 4 — High-Risk Order Profile

**Insight from data:** Orders with `price > 200` are 40% more likely to be late. Orders with `estimated_days <= 10` have the highest late rate (10.5%) — likely because short windows leave no buffer. This creates a risk score per order.

In [0]:
# Price distribution
price_stats = df.select("price").collect()
prices = [row.price for row in price_stats if row.price is not None]

price_q1 = statistics.quantiles(prices, n=4)[0]
price_q2 = statistics.quantiles(prices, n=4)[1]
price_q3 = statistics.quantiles(prices, n=4)[2]

print(f"Price Q1: {price_q1}, Q2: {price_q2}, Q3: {price_q3}")

# Delivery days distribution
days_stats = df.select("estimated_days").collect()
days = [row.estimated_days for row in days_stats if row.estimated_days is not None]

days_q1 = statistics.quantiles(days, n=4)[0]
days_q2 = statistics.quantiles(days, n=4)[1]
days_q3 = statistics.quantiles(days, n=4)[2]

print(f"Days Q1: {days_q1}, Q2: {days_q2}, Q3: {days_q3}")

Price Q1: 39.9, Q2: 74.99, Q3: 134.9
Days Q1: 19.0, Q2: 24.0, Q3: 29.0


In [0]:
agg_risk = (
    df
    .withColumn("price_tier",
        F.when(F.col("price") <= price_q1, "Low")
         .when(F.col("price") <= price_q2, "Mid")
         .when(F.col("price") <= price_q3, "High")
         .otherwise("Premium")
    )
    .withColumn("delivery_window",
        F.when(F.col("estimated_days") <= days_q1, "Tight")
         .when(F.col("estimated_days") <= days_q2, "Normal")
         .when(F.col("estimated_days") <= days_q3, "Extended")
         .otherwise("Long")
    )
    .groupBy("price_tier", "delivery_window")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("is_late").alias("late_orders"),
        F.round(F.avg("is_late") * 100, 2).alias("late_rate_pct"),
        F.round(F.avg("price"), 2).alias("avg_price")
    )
    .orderBy(F.col("late_rate_pct").desc())
)

display(agg_risk)

price_tier,delivery_window,total_orders,late_orders,late_rate_pct,avg_price
Premium,Normal,6831,607,8.89,291.93
Mid,Normal,7342,612,8.34,56.19
Premium,Extended,6269,519,8.28,302.42
Premium,Tight,5798,454,7.83,294.4
High,Normal,7323,555,7.58,101.55
Mid,Extended,6086,445,7.31,56.49
Low,Normal,7075,511,7.22,26.04
High,Extended,6276,438,6.98,101.94
High,Tight,6446,433,6.72,100.96
Premium,Long,7584,458,6.04,309.17


In [0]:
write_gold(agg_risk, "price_delivery_risk_matrix")

✓ Saved project.gold.price_delivery_risk_matrix  (16 rows)


---
## 7. Aggregation 5 — Model Predictions vs Actual (from Predictions Table)

Uses `delivery_delay_predictions` (the ML output table) to summarize model performance — how well predicted late deliveries match actual ones, by state and category.

In [0]:
# from pyspark.ml.functions import vector_to_array
# from pyspark.sql.window import Window

# # 1. Extract probability and create risk percentile
# new_pred_df = pred_df.withColumn(
#     "late_probability",
#     vector_to_array(F.col("probability"))[1].cast("double")
# ).withColumn(
#     "risk_percentile",
#     F.percent_rank().over(Window.orderBy(F.col("late_probability").desc()))
# ).withColumn(
#     "prediction_adjusted",
#     F.when(F.col("risk_percentile") <= 0.1, 1).otherwise(0)
# )

# display(new_pred_df.select("late_probability", "risk_percentile", "prediction_adjusted").limit(3))


In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings("ignore")

# Extract probability
new_pred_df = pred_df.withColumn(
    "late_probability",
    vector_to_array(F.col("probability"))[1].cast("double")
)

# Get 90th percentile threshold (top 10%)
threshold_90 = new_pred_df.approxQuantile("late_probability", [0.9], 0.01)[0]

# Flag top 10% without window function
new_pred_df = new_pred_df.withColumn(
    "prediction_adjusted",
    F.when(F.col("late_probability") >= threshold_90, 1).otherwise(0)
)

display(new_pred_df.select("late_probability", "prediction_adjusted").limit(3))

late_probability,prediction_adjusted
0.5423625180504472,0
0.31958455768382105,0
0.4336842650399951,0


In [0]:
# Model performance by state using probability
agg_model_state = (
    new_pred_df.groupBy("customer_state")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("is_late").alias("actual_late"),
        F.sum("prediction_adjusted").alias("predicted_late_top10pct"),
        F.round(F.avg("late_probability") * 100, 2).alias("avg_late_probability_pct"),
        F.sum(
            F.when(F.col("is_late") == F.col("prediction_adjusted"), 1).otherwise(0)
        ).alias("correct_predictions")
    )
    .withColumn("accuracy_pct",
        F.round(F.col("correct_predictions") / F.col("total_orders") * 100, 2)
    )
    .withColumn("actual_late_rate_pct",
        F.round(F.col("actual_late") / F.col("total_orders") * 100, 2)
    )
    .withColumn("predicted_late_pct",
        F.round(F.col("predicted_late_top10pct") / F.col("total_orders") * 100, 2)
    )
    .orderBy(F.col("avg_late_probability_pct").desc())
)

# 3. Overall model performance
agg_model_overall = new_pred_df.agg(
    F.count("*").alias("total_predictions"),
    F.sum("is_late").alias("total_actual_late"),
    F.sum("prediction_adjusted").alias("total_predicted_late_top10pct"),
    F.round(F.avg("late_probability") * 100, 2).alias("avg_late_probability_pct"),
    F.round(
        F.sum(F.when(F.col("is_late") == F.col("prediction_adjusted"), 1).otherwise(0))
        / F.count("*") * 100, 2
    ).alias("accuracy_top10pct_precision")
)

print("=== Overall Model Performance ===")
display(agg_model_overall)

print("=== Model Performance by State ===")
display(agg_model_state)


=== Overall Model Performance ===


total_predictions,total_actual_late,total_predicted_late_top10pct,avg_late_probability_pct,accuracy_top10pct_precision
21429,1459,2331,44.09,87.19


=== Model Performance by State ===


customer_state,total_orders,actual_late,predicted_late_top10pct,avg_late_probability_pct,correct_predictions,accuracy_pct,actual_late_rate_pct,predicted_late_pct
RJ,2736,312,915,53.03,1961,71.67,11.4,33.44
BA,702,82,157,53.0,531,75.64,11.68,22.36
MA,152,31,52,52.75,109,71.71,20.39,34.21
CE,320,49,80,51.42,251,78.44,15.31,25.0
AL,91,19,17,51.15,71,78.02,20.88,18.68
MS,176,13,46,50.42,133,75.57,7.39,26.14
SC,851,72,176,49.19,675,79.32,8.46,20.68
PB,112,14,16,48.52,86,76.79,12.5,14.29
ES,427,54,63,48.45,340,79.63,12.65,14.75
TO,80,8,16,48.28,64,80.0,10.0,20.0


In [0]:

write_gold(agg_model_overall, "agg_model_overall_performance")
write_gold(agg_model_state,   "agg_model_performance_by_state", zorder_col="customer_state")

✓ Saved project.gold.agg_model_overall_performance  (1 rows)
✓ Saved project.gold.agg_model_performance_by_state  (27 rows)


---
## 8. Summary — Gold Tables Created

| Table | Feeds Dashboard Chart |
|---|---|
| `agg_late_by_state` | Late rate by state (geographic risk map) |
| `agg_late_by_category` | Late rate + revenue by category |
| `agg_late_by_month` | Seasonal late rate trend |
| `agg_high_risk_profiles` | Price tier × delivery window risk matrix |
